# 00 - Scope and local setup

- **Meeting item:** Approval gate
- **Commit:** C01 - `chore: add guarded package and test scaffold`
- **Commit achievement:** establishes a local, synthetic-only workspace with an explicit blocked approval state.
- **Data mode:** `synthetic` (no dataset matrix is read in this notebook)
- **Evidence labels used:** verified, unknown
- **Human gate:** can a reviewer see runtime, scope, approval state, and the next decision on one page?
- **Stop condition:** missing Python 3.11, unclear approval state, or unexplained dirty files.

This notebook only reports facts collected from the local machine and the plan records under `plan/`.
All logic it calls lives in `src/p22` and `scripts/plan_guard.py`.

## 1. Runtime facts (evidence label: verified)

CPU is the supported default. Accelerator availability is reported, never assumed.

In [ ]:
import platform
import sys
from importlib.metadata import version

import torch

import p22

REPO_ROOT = p22.REPO_ROOT

print(f"repository root: {REPO_ROOT}")
print(f"python: {platform.python_version()} ({sys.executable})")
print(f"platform: {platform.platform()}")
print(f"p22 version: {p22.__version__}")
for package in ("numpy", "pandas", "scikit-learn", "scipy", "torch", "anndata"):
    print(f"{package}: {version(package)}")
print(f"torch default device: cpu")
print(f"torch mps available: {torch.backends.mps.is_available()}")
print(f"torch cuda available: {torch.cuda.is_available()}")

## 2. Repository and plan state (evidence label: verified)

In [ ]:
import subprocess


def git(*args: str) -> str:
    result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args],
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.strip()


print(f"branch: {git('branch', '--show-current')}")
print(f"head: {git('rev-parse', 'HEAD')}")
print("recent history:")
print(git("log", "--format=%h %s", "-3"))
dirty = git("status", "--porcelain=v1", "-uall")
print("working tree changes (expected: only C01 allowlisted paths before commit):")
print(dirty or "<clean>")

In [ ]:
status = subprocess.run(
    [sys.executable, "scripts/plan_guard.py", "status"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(status.stdout)

## 3. Approval state and scope (evidence label: verified)

Everything condition-specific is blocked until the named approver records a decision.
The package refuses a data mode that the approval record does not allow.

In [ ]:
approvals = p22.load_approvals()
print(f"approval state: {approvals['approval_state']}")
print(f"approver: {approvals['approver']}")
print(f"allowed data modes: {p22.allowed_data_modes(approvals)}")
print(f"approval blocked: {p22.approval_blocked(approvals)}")
print("")
print("allowed before approval:")
for item in approvals["allowed_before_approval"]:
    print(f"  - {item}")
print("forbidden before approval:")
for item in approvals["forbidden_before_approval"]:
    print(f"  - {item}")

In [ ]:
print(f"accepted: {p22.require_allowed_data_mode(p22.SYNTHETIC, approvals)!r}")
try:
    p22.require_allowed_data_mode(p22.APPROVED_REAL_DATA, approvals)
except ValueError as error:
    print(f"rejected as expected: {error}")
else:
    raise AssertionError("blocked data mode was accepted")

## 4. Gate checks

These assertions fail the notebook if the stop conditions for C01 are met.

In [ ]:
import json

state = json.loads((REPO_ROOT / "plan" / "state.json").read_text())
plan = json.loads((REPO_ROOT / "plan" / "implementation_plan.json").read_text())

assert sys.version_info[:2] == (3, 11), sys.version_info
assert git("branch", "--show-current") == plan["branch"]
assert p22.approval_blocked(approvals), "approval state must remain blocked"
assert state["data_mode"] == p22.SYNTHETIC
assert len(plan["steps"]) == 13
tracked_data = git("ls-files", "data").splitlines()
assert tracked_data in ([], ["data/README.md"]), tracked_data
print("all C01 gate checks passed")
print(f"next action recorded in plan/state.json: {state['next_action']}")

## 5. Unknowns (evidence label: unknown)

- Whether any condition-specific dataset will be approved, and which one.
- Whether paired two-modality data with donor identifiers will be available for that dataset.
- Accelerator availability on the reviewer's machine; CPU remains the supported default.

## Gate

**PASS**

- **Observed:** local Python 3.11 runtime with pinned scientific packages, branch `develop`, plan records for 13 steps, approval state blocked, and a guard that rejects a non-allowed data mode.
- **Remains unknown:** the approved dataset, target, and modality pairing; no dataset matrix is tracked in this repository.
- **Next decision owner:** researcher continues with C02 (legacy evidence boundary). Condition-specific work stays blocked pending the named approver.